# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one content item (one page) observed over a fixed 90-day window.

One row = one page, identified by content_hash_id, belonging to one client
(client_hash_id). The time window I will use is month=2026-03 (March 2026)
from the fact_content_daily_performance table — a mid-panel month that avoids
the sealed final month (June 2026).

I will aggregate daily rows up to one row per content item by summing or
averaging the daily metrics across the month.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
subprocess.run(["pip", "install", "huggingface_hub", "duckdb", "-q"], check=True)

import duckdb
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta; LOAD delta;")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

# Verify grain
result = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_hash_id) as unique_pages,
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pages earliest_date latest_date
0     9841378        331437    2026-03-01  2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Fields sorted into four buckets:

FEATURES (knowable at decision moment):
- impressions: total search impressions in the month — knowable before any
  review decision is made
- clicks: total clicks in the month — knowable before any review decision
- avg_position: average search ranking position — knowable before any decision
- ctr: click-through rate (clicks / impressions) — derived from observed signals,
  knowable before any decision
- content_age_days: how old the page is — knowable before any decision

LABEL / PROXY:
- is_declining: pages where impressions dropped compared to a prior window —
  this is the proxy target I will define from observed trend direction

CONTEXT (used for grouping and filtering, not as model features):
- client_hash_id: identifies which client the page belongs to — used for
  client-holdout validation split
- content_hash_id: unique page identifier — used for joining and deduplication
- report_date: used to define the time window, not a feature

EXCLUDED:
- sessions_ai / ai_traffic_pct: excluded because AI-referral rows are very
  sparse (only 30,177 rows out of 78M) — including them would introduce
  noise rather than signal for this lane
- Any product decision flags (health_score, priority_score): not in the
  dataset by design — excluded to avoid circular results

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the available columns in the table
cols = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'fact_content_daily_performance'
""").df()

# Alternative if above doesn't work
sample = con.execute("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""").df()
print("Columns available:")
print(list(sample.columns))
print("\nSample row:")
print(sample.T)

Columns available:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Sample row:
                                                 0
report_date                    2026-03-01 00:00:00
client_hash_id             client_73cda7b4e4f265ea
content_hash_id           content_b7e512995f79d5a6
client_has_gsc                                True
client_has_ga4                               False
gsc_data_available                            True
ga4_data_available                            <NA>
gsc_impressions          

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries:

Query 1 — Grain check: confirm one row per content item per day
(total rows should equal unique content + date combinations)

Query 2 — Row count and date span: confirmed above — 9,841,378 rows,
331,437 unique pages, March 1 to March 31 2026

Query 3 — Availability check: how many rows have GSC impressions data
(gsc_data_available IS TRUE)

In [12]:
# Five feature frame using correct column names
features = con.execute("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE 0 END as ctr,
        COUNT(DISTINCT report_date) as days_with_data
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_impressions > 0
    GROUP BY content_hash_id, client_hash_id
    LIMIT 10
""").df()
print("\nFive-feature frame (first 10 rows):")
print(features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Five-feature frame (first 10 rows):
            content_hash_id           client_hash_id  total_impressions  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea               57.0   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea             6523.0   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea              149.0   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea              453.0   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea             5630.0   
5  content_05434271b257bb68  client_73cda7b4e4f265ea             1421.0   
6  content_22610b0934f8825e  client_73cda7b4e4f265ea               67.0   
7  content_712c365258cee05c  client_73cda7b4e4f265ea             6048.0   
8  content_5d412fba6e1a2582  client_73cda7b4e4f265ea              223.0   
9  content_1f380a642aed423b  client_73cda7b4e4f265ea               96.0   

   total_clicks  avg_position       ctr  days_with_data  
0           0.0      2.714744  0.000000              26  
1           7.0      

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits and named limitations:

1. Unbalanced history: different clients have different amounts of history.
   Some clients have data starting from early 2025, others joined later.
   This means a "30-day decline" means something different depending on
   when a client started — directionally observed, not uniformly measured.

2. GSC-only early rows: rows from before a client's GA4 tracking started
   contain search data only. Sessions and engagement metrics are absent for
   those rows. Using engagement features without filtering for ga4_data_available
   would introduce silent missing values that could mislead the model.

3. Proxy label weakness: the label I am using (declining trend in the current
   window) is not a future-looking outcome. It is calculated from the same
   window as the features, which means the model is not truly predicting
   the future — it is describing the present. A stronger label would require
   a prior feature window and a separate future target window.

4. Volume threshold needed: with 331,437 unique pages in March 2026 alone,
   many pages have very low impressions — one or two per month. These are
   noise, not signal. A minimum impressions filter will be required before
   modeling to keep only pages with enough data to make a meaningful
   recommendation.

In [13]:
threshold_check = con.execute("""
    SELECT
        SUM(CASE WHEN total_impressions >= 100 THEN 1 ELSE 0 END) as pages_above_100_impressions,
        SUM(CASE WHEN total_impressions >= 500 THEN 1 ELSE 0 END) as pages_above_500_impressions,
        COUNT(*) as total_pages
    FROM (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) as total_impressions
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions > 0
        GROUP BY content_hash_id
    )
""").df()
print("Pages surviving minimum impressions thresholds:")
print(threshold_check)

Pages surviving minimum impressions thresholds:
   pages_above_100_impressions  pages_above_500_impressions  total_pages
0                     101441.0                      61924.0       176738


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.